## **집계연산**

### **PySpark 설정**

In [1]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext

In [2]:
# [+] SparkConf, SparkContext 객체 생성하기
conf = SparkConf()

In [3]:
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 04:14:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **reduce( ) 메서드**

In [4]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x,y: x+y)

15

In [5]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()

[1, 2, 3, 4, 5]

In [6]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [7]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
rdd.reduce(lambda x, y: x+y)

15

In [8]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5]).reduce(lambda x, y: (x*2) + y)

35

In [9]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

In [10]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 3).reduce(lambda x,y: (x * 2) + y)

31

In [11]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 4).reduce(lambda x,y: (x * 2) + y)

35

### **10000 개 정수 난수의 총합 계산하기**

In [12]:
!pip install numpy

In [13]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [14]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0, 100, size=10000)
x[:100]

array([85, 48, 97, 88, 30, 61, 70, 50,  8,  1, 52, 21, 47,  9, 14, 38,  9,
       66, 19,  6, 96, 60, 87, 31, 98, 75, 79, 20, 46, 70, 78, 60,  8, 90,
       37, 39, 36, 98, 48, 88,  0, 53, 12, 46, 34, 71, 11,  5, 62, 42,  7,
       44,  3, 68, 49,  7, 25, 29, 93, 80, 19, 66, 68,  0, 30, 24,  1, 12,
       81, 16, 85, 27, 84, 58, 98, 10, 29, 24, 40, 53, 51, 14,  7,  3, 60,
       70, 83, 41, 52,  1, 77, 83, 27, 16,  6, 90, 74, 64, 30, 52])

In [15]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x, 8)

In [16]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

Partition 0: [np.int64(85), np.int64(48), np.int64(97), np.int64(88), np.int64(30), np.int64(61), np.int64(70), np.int64(50), np.int64(8), np.int64(1), np.int64(52), np.int64(21), np.int64(47), np.int64(9), np.int64(14), np.int64(38), np.int64(9), np.int64(66), np.int64(19), np.int64(6), np.int64(96), np.int64(60), np.int64(87), np.int64(31), np.int64(98), np.int64(75), np.int64(79), np.int64(20), np.int64(46), np.int64(70), np.int64(78), np.int64(60), np.int64(8), np.int64(90), np.int64(37), np.int64(39), np.int64(36), np.int64(98), np.int64(48), np.int64(88), np.int64(0), np.int64(53), np.int64(12), np.int64(46), np.int64(34), np.int64(71), np.int64(11), np.int64(5), np.int64(62), np.int64(42), np.int64(7), np.int64(44), np.int64(3), np.int64(68), np.int64(49), np.int64(7), np.int64(25), np.int64(29), np.int64(93), np.int64(80), np.int64(19), np.int64(66), np.int64(68), np.int64(0), np.int64(30), np.int64(24), np.int64(1), np.int64(12), np.int64(81), np.int64(16), np.int64(85), np.in

In [17]:
# [+] x+y 집계 연산
rdd.reduce(lambda x,y: x+y)

np.int64(493222)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [18]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x, y: x+y)

15

In [19]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4], 2).fold(1, lambda x, y: x+y)

13

In [20]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x,y : x+y)

10

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [26]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([3, 10, 7, 5, 12, 8, 10]).groupBy(lambda x: x%2).collect()

In [27]:
# [+] groupBy() 결과 출력하기
for k, v in res:
    print(k, list(v))

0 [10, 12, 8, 10]
1 [3, 7, 5]


### **aggregate( ) 메서드**

In [28]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4])\
    .reduce(lambda x, y: (x + y, x * y)) 
    # single value -> pair 생성 시도할 경우 오류

TypeError: can only concatenate tuple (not "int") to tuple

In [29]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

TypeError: unsupported operand type(s) for +: 'int' and 'tuple'

In [30]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [31]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)

(10, 4)